In [1]:
import torch
from torch import nn

#from encoder_decoder import Encoder
#from encoder_decoder import Decoder

class Transformer(nn.Module):
    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        hidden_dim: int,
        n_layers: int,
        n_heads: int,
        forward_expansion: int,
        dropout: float,
        src_pad_idx: int,
        trg_pad_idx: int,
        device: torch.device,
        max_len: int = 256
    ):
        super().__init__()

        self.encoder = Encoder(
            input_dim = input_dim,
            hidden_dim = hidden_dim,
            n_layers = n_layers,
            n_heads = n_heads,
            forward_expansion = forward_expansion,
            dropout = dropout,
            device = device,
            pad_idx = src_pad_idx,
            max_length = max_length
        )

        self.decoder = Decoder(
            output_dim = output_dim,
            hidden_dim = hidden_dim,
            n_layers = n_layers,
            n_heads = n_heads,
            forward_expansion = forward_expansion,
            dropout = dropout,
            device = device,
            pad_idx = trg_pad_idx,
            max_length = max_length
        )

        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx

        self.device = device

    def make_src_mask(self, src):
        return (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2).to(self.device)

    def make_trg_mask(self, trg):
        # создание матрицы - состоящей из единиц на главной диагонали и ниже - запрещаем смотреть на правые токены
        trg_mask = torch.tril(
            torch.ones((trg.shape[1], trg.shape[1]), device = self.device)
        ).bool()
        trg_mask = (trg != self.trg_pad_idx).unaqueeze(1).unsqueeze(2) & trg_mask
        return trg_mask.to(self.device)

    def forward(self, src, trg):
        src_mask = make_src_mask(src)
        trg_mask = make_trg_mask(trg)

        return self.decoder(trg, self.encoder(src, src_mask), src_mask, trg_mask)

In [2]:
#from attention import Attention
def get_encoding(hidden_dim, max_length):
    pe = np.array(
        [
            [i / np.power(10000, 2 * (j//2) / hidden_dim) for j in range(hidden_dim)]
            for i in range(max_length)
        ]
    )
    pe[:, 0::2] = np.sin(pe[:, 0::2])
    pe[:, 1::2] = np.cos(pe[:, 1::2])

    return torch.FloatTensor(pe).unsqueeze(0)

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, hidden_dim, max_length = 100):
        super().__init__()

        self.register_buffer('pe', get_encoding(hidden_dim, max_length))
        self.pe = self.pe.to(torch.device('cuda:0'))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [4]:
class EncoderLayer(nn.Module):
    def __init__(self,
                hidden_dim: int,
                n_heads: int,
                dropout: float,
                forward_expansion: int
                ):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.attention = Attention(hidden_dim, n_heads, dropout)

        self.feed_forward = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * forward_expansion),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * forward_expansion, hidden_dim)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.norm1(x + self.dropout(self.attention(x, x, x, mask)))

        return self.norm2(x + self.dropout(self.feed_forward(x)))

In [5]:
class Encoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        n_layers: int,
        n_heads: int, 
        forward_expansion: int,
        dropout: float,
        device: torch.device,
        pad_idx: int,
        max_length: int = 90
    ):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, hidden_dim, padding_index = pad_idx)
        self.positional_encoding = PositionalEncoding(hidden_dim, max_length)

        self.layers = nn.ModuleList(
            [
                EncoderLayer(
                    hidden_dim = hidden_dim,
                    n_heads = n_heads,
                    dropout = dropout,
                    forward_expansion = forward_expansion
                )
                for _ in range(n_layers)
            ]
        )
        self.scale = hidden_dim ** 0.5
        self.dropout = nn.Dropout(dropout)
        self.device = device

    def forward(self, x, mask):
        x = self.dropout(self.positional_encoding(self.embedding(x) * self.scale))

        for layer in self.layers:
            x = layer(x, mask)

        return x
        

In [7]:
class DecoderLayer(nn.Module):
    def __init__(self,
                hidden_dim: int,
                n_heads: int, 
                dropout: float,
                forward_expansion: int,
                ):
        super().__init__()
        
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)

        self.attention1 = Attention(hidden_dim, n_heads, dropout)
        self.attention2 = Attention(hidden_dim, n_heads, dropout)

        self.feed_dorward = nn.Sequential(
            nn.Linear(in_features = hidden_dim, out_features = hidden_dim * forward_expansion),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(in_features = hidden_dim * forward_expansion, out_features = hidden_dim)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_enc, src_mask, trg_mask):
        x = self.norm1(x + self.dropout(self.attention1(x, x, x, trg_mask)))
        x = self.norm2(x + self.dropout(self.attention2(x, src_enc, src_enc, src_mask)))
        return self.norm3(x + self.dropout(self.feed_forward(x)))
        

In [8]:
class Decoder(nn.Module):
    def __init__(
        self,
        output_dim: int,
        hidden_dim: int,
        n_layers: int,
        n_heads: int,
        forward_expansion: int,
        dropout: float,
        device: torch.device,
        pad_idx: int,
        max_length: int = 90
    ):
        super().__init__()

        self.embedding = nn.Embedding(output_dim, hidden_dim, padding_idx = pad_idx)
        self.positional_encoding = PositionalEncoding(hidden_dim, max_length)

        self.layers = nn.ModuleList([
            DecoderLayer(
                hidden_dim = hidden_dim,
                n_heads = n_heads,
                dropout = dropout,
                forward_expansion = forward_expansion
            )
            for _ in range(n_layers)
        ])

        self.fc = nn.Linear(hidden_dim, output_dim)

        self.scale = hidden_dim ** 0.5
        self.dropout = nn.Dropout(dropout)
        self.device = device

    def forward(self, x, enc_out, src_mask, trg_mask):
        x = self.dropout(self.positional_encoding(self.embedding(x) * self.scale))

        for layer in self.layers:
            x = layer(x, enc_out, src_mask, trg_mask)

        return self.fc(x)

In [10]:
BIAS: bool = True

class Attention(nn.Module):
    def __init__(self, 
                hidden_dim: int,
                n_heads: int,
                dropout: float):
        super().__init__()
        assert(
            hidden_dim % n_heads == 0
        ), "Hidden size should be divisible by number of heads"

        self.hidden_dim = hidden_dim
        self.n_heads = n_heads
        self.head_dim = hidden_dim // n_heads

        self.scale = hidden_dim ** (1 / 2)

        self.q = nn.Linear(hidden_dim, hidden_dim, bias = BIAS)
        self.k = nn.Linear(hidden_dim, hidden_dim, bias = BIAS)
        self.v = nn.Linear(hidden_dim, hidden_dim, bias = BIAS)

        self.fc = nn.Linear(hidden_dim, hidden_dim, bias = BIAS)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask = None):
        # прокинем skip-connection
        residual = q

        batch_size = q.shape[0]

        # матрицы wq, wk, wv
        q = self.q(q).reshape(batch_size, -1, self.n_heads, self.head_dim)
        k = self.k(k).reshape(batch_size, -1, self.n_heads, self.head_dim)
        v = self.v(v).reshape(batch_size, -1, self.n_heads, self.head_dim)

        # перемножим матрицы q*k^T
        energy = torch.einsum('nqhd,nkhq->nhqk', q, k)
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e9)

        # получаем скоры от нормальзованного произведения
        attention = torch.softmax(energy / self.scale, dim = 3)

        # умножаем на v
        out = torch.einsum('nhqk,nkhd->nqhd', self.dropout(attention), v).reshape(
            batch_size, -1, self.hidden_dim
        )
        # нормализуем и прокидываем skip connection
        return self.norm(self.fc(out) + residual)

In [11]:
def get_encoding(hidden_size, max_length):
    pe = np.array([
        [
            i / np.power(10000, 2 * (j // 2) / hidden_size) for j in range(hidden_size)
        ] for i in range(max_length)
    ])
    pe[:, 0::2] = np.sin(pe[:, 0::2])
    pe[:, 1::2] = np.cos(pe[:, 1::2])

    return torch.FloatTensor(pe).unsqueeze(0)

class PositionalEncoding(nn.Module):
    def __init__(self, hidden_size, max_length = 512):
        super().__init__()

        self.register_buffer('pe', get_encoding(hidden_size, max_length))
        self.pe = self.pe.to(torch.device('cuda:0'))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [13]:
class AnekdotesModel(nn.Module):
    def __init__(self, num_tokens: int, hidden_size: int = 128):
        super().__init__()

        self.embedding = nn.Embedding(num_tokens, hidden_size, padding_idx = pad_idx)
        self.positional_encoding = PositionalEncoding(hidden_size)

        self.transformer = nn.TransformerEncoder(
            encoder_layer = nn.TransformerEncoderLayer(
                d_model = hidden_size,
                nhead = 8,
                dim_feedforward = hidden_size,
                batch_first = True,
                dropout = 0.1
            ),
            num_layers = 6
        )

        self.output = nn.Linear(hidden_size, num_tokens)

        self.dropout = nn.Dropout(0.1)

        self.scale = hidden_size ** 0.5
        self.pad_idx = pad_idx

    def forward(self, x):
        # маска - матрица с единицами в левой нижней диагонали - не позволяет смотреть в следующие токены
        mask = nn.Transformer.generate_square_subsequent_mask(x.shape[1]).to(device)
        # маска - не смотрим на паддинг
        key_mask = x == self.pad_idx

        x = self.dropout(self.positional_encoding(self.embedding(x) * self.scale))
        x = self.transformer(x, mask = mask, src_key_padding_mask = key_mask)
        x = self.output(x)

        return x

### Подготовим данные

In [14]:
aneks = []

with open('anek.txt', 'r', encoding = 'utf-8') as f:
    aneks.extend(f.read().split('\n'))

In [15]:
class Vocabulary:
    def __init__(self):
        self.vocabulary = ['_', '#', '<'] # start, end, padding
        self.vocabulary.extend('абвгдеёжзийклмнопрстуфхцчшщъыьэюя')
        self.vocabulary.extend('абвгдеёжзийклмнопрстуфхцчшщъыьэюя'.upper())
        self.vocabulary.extend('0123456789 ,.!?-')

        self._idx2char = {i: char for i, char in enumerate(self.vocabulary)}
        self._char2idx = {char: i for i, char in enumerate(self.vocabulary)}

    def get_vocabulary(self):
        return self.vocabulary

    def idx2char(self, idx: int):
        if idx not in self._idx2char:
            return '<'
        return self._idx2char[idx]

    def char2idx(self, char: str):
        if char not in self._char2idx:
            return self.get_pad()
        return self._char2idx[char]

    def encode(self, text):
        result = [self.char2idx(char) for char in text]
        result = [self.get_sos()] + result + [self.get_eos()]
        return result

    def get_sos(self):
        return self.char2idx('_')

    def get_eos(self):
        return self.char2idx('#')

    def get_pad(self):
        return self.char2idx('<')
    

In [16]:
from torch.utils.data import Dataset

class Anekdotes(Dataset):
    def __init__(self, aneks):
        self.aneks = aneks
        self.vocab = Vocabulary()

    def __getitem__(self, idx):
        return torch.LongTensor(self.vocab.encode(self.aneks[idx]))

    def __len__(self):
        return len(self.aneks)

In [17]:
dataset = Anekdotes(aneks)

In [18]:
from torch.utils.data import random_split

train_dataset, valid_dataset = random_split(dataset, (int(len(dataset) * 0.9), len(dataset) - int(len(dataset) * 0.9)))

In [19]:
from torch.nn.utils.rnn import pad_sequence
pad_idx = dataset.vocab.get_pad()

def collate_fn(batch):
    return pad_sequence([b[:256] for b in batch], padding_value = pad_idx, batch_first = True)

In [20]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size = 256, collate_fn = collate_fn, pin_memory = True, num_workers = 4)
valid_loader = DataLoader(valid_dataset, batch_size = 256, collate_fn = collate_fn, pin_memory = True, num_workers = 4)

### Напишем функции для обучения

In [23]:
import numpy as np
model = AnekdotesModel(num_tokens = len(dataset.vocab.get_vocabulary()))

AssertionError: Torch not compiled with CUDA enabled

In [25]:
from tqdm import tqdm
def train(model):
    model.train()
    total_loss = 0
    for x in tqdm(train_loader, desc = 'Train'):
        x = x.to(device)
        optimizer.zero_grad()
        
        output = model(x[:, :-1]).transpose(1, 2)
        
        loss = loss_fn(output, x[:, 1:])
        
        total_loss += loss.item()
        
        loss.backward()
        
        optimizer.step()
    total_loss /= len(train_loader)
    return total_loss

In [26]:
@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    for x in tqdm(loader, desc = 'Evaluation'):
        x = x.to(device)
        
        output = model(x[:, :-1]).transpose(1, 2)
        
        loss = loss_fn(output, x[:, 1:])
        
        total_loss += loss.item()
    total_loss /= len(loader)
    return total_loss

In [27]:
def plot_stats(
    train_loss: list[float],
    valid_loss: list[float],
    title: str
):
    plt.figure(figsize = (16, 10))
    plt.title(title)
    plt.plot(train_loss, label = 'Train loss')
    plt.plot(valid_loss, label = 'Valid loss')
    plt.legend()
    plt.grid()
    plt.show()

In [31]:
from IPython.display import clear_output
def whole_train_value_cycle(
    model,
    num_epochs: int,
    title: str
):
    train_loss_history, valid_loss_history = [], []

    for epoch in range(num_epochs):
        train_loss = train(model)
        valid_loss = evaluate(model, valid_loader)

        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)

        clear_output(wait = True)

        plot_stats(
            train_loss_history,
            valid_loss_history,
            title
        )

### Обучим модель

In [32]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [33]:
from torch.optim import Adam
model = model.to(device)
optimizer = Adam(model.parameters(), lr = 2e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index = pad_index)
wwhole_train_value_cycle(model, 10, 'Anekdote Transformer')

NameError: name 'model' is not defined